In [1]:
from langfuse import observe, propagate_attributes
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
import uuid
from unittest import result

import dotenv

In [3]:
import socket
from src.examples.integrateions.agent_tools import get_local_ip
dotenv.load_dotenv(override=True)

model = ChatOpenAI(
    model_name="MiniMax-M2.7",
    api_key=os.getenv("MINIMAX_API_KEY"),
    base_url=os.getenv("MINIMAX_BASE_URL"),
)

ipv4 = get_local_ip(socket.AF_INET)
print(f"IPv4: {ipv4}")

IPv4: 28.0.0.1


In [9]:

@observe(name="user-query-processing") # Automatically log function as a trace to Langfuse
def process_user_query(user_input: str):
    # Propagate trace attributes to all child observations
    with propagate_attributes(
        trace_name="user-query-processing-v1", #  比 @observe(name="user-query-processing") 优先
        session_id="session-1234",
        user_id="user-5678",
    ):

      # Initialize the Langfuse handler - automatically inherits the current trace context
      langfuse_handler = CallbackHandler()

      # Your Langchain code - will be nested under the @observe trace
      prompt = ChatPromptTemplate.from_template("Respond to: {input}")
      chain = prompt | model

      result = chain.invoke({"input": user_input}, config={"callbacks": [langfuse_handler]})

    return result.content

# Usage
answer = process_user_query("What is the capital of France?")
print(answer)


<think>
The user asks: "Respond to: What is the capital of France?" They want a direct answer. The question is straightforward. It's likely a simple factual answer. We must ensure compliance with policy. There's no disallowed content. Just answer: Paris.

We could also add a brief note if needed.

Thus final: "Paris."
</think>

Paris.
